# SO-101: from contact grasping to VLA evaluation

Run this notebook on the Linux RTX GPU host after completing the module README build and license steps. It uses the same Docker CLI as the terminal workflow. The recorded baseline with the original yellow robot is expert 20/20 and SmolVLA 0/20 on RTX 3060; consult `VALIDATION.md` for installation checks, configuration and limitations.

Objectives: inspect RGB observations, collect successful expert demonstrations, read LeRobot data, and measure a pretrained policy on a fixed benchmark.

In [ ]:
from pathlib import Path
import json, os, subprocess
from IPython.display import Image, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts/run.sh").exists())
assert os.environ.get("ACCEPT_EULA") == "Y", "Read the NVIDIA license, then set ACCEPT_EULA=Y before starting Jupyter."

RUNNER = ROOT / os.environ.get("SO101_RUNNER", "scripts/run.sh")

def lab(*args):
    subprocess.run([str(RUNNER), *map(str, args)], cwd=ROOT, check=True)

lab("doctor")

## Inspect the environment

Downloads are public, revision-pinned and checksum-verified. Two cameras observe the same physical scene. The VLA does not receive cube or target coordinates. Choose a new output name if you already ran these cells.

In [ ]:
lab("fetch")
lab("convert")
lab("check-kinematics")
lab("check-scene", "--output", "runs/notebook-scene")
for camera in ("top", "wrist"):
    display(Image(filename=str(ROOT / f"runs/notebook-scene/{camera}.png")))

## Validate the expert and collect demonstrations

First require all 20 benchmark episodes to succeed using real finger contacts. Collection uses disjoint seeds from 10000 onward. Inspect failures as carefully as successes; only successful collection episodes become training data.

In [ ]:
lab("evaluate", "--policy", "expert", "--output", "runs/notebook-expert", "--video")
lab("audit", "--run", "runs/notebook-expert")
lab("collect", "--seed", 10000, "--episodes", 10, "--output", "runs/notebook-demos", "--video")
lab("export", "--run", "runs/notebook-demos", "--dataset", "runs/notebook-dataset")
lab("check-dataset", "--dataset", "runs/notebook-dataset")

## Evaluate SmolVLA

The checkpoint was trained in a different IsaacLab scene. Preserve the camera mapping, joint calibration, 10 Hz control and saved processors. A zero SR is a valid experimental outcome. Inference and simulation run in separate processes; simulation time pauses while waiting for an action.

In [ ]:
lab("policy-smoke")
lab("evaluate", "--policy", "smolvla", "--output", "runs/notebook-smolvla", "--video")
lab("audit", "--run", "runs/notebook-smolvla")

In [ ]:
for name in ("expert", "smolvla"):
    result = json.loads((ROOT / f"runs/notebook-{name}/run.json").read_text())
    assert result["complete"], "Execution errors invalidate the SR."
    print(f"{name}: {result['successes']}/{result['denominator']} = {result['success_rate']:.1%}")
    print("Configuration:", result["config_sha256"])

## Project exercise

Copy `configs/default.json` to `configs/project.json`, change one scene or camera parameter, rebuild, and pass `--config configs/project.json`. Repeat expert and policy evaluation under a new output directory. Keep all seeds.

Explain: (1) why hovering above the target must fail; (2) why joint radians cannot be fed directly to this checkpoint; (3) what a scene change does to the interpretation of SR; (4) how successful-only demonstration export differs from unbiased evaluation. Use the saved physics traces, diagnostic videos and GPU memory samples as evidence.